# Create Genome BC Awards (Genome British Columbia)

Creates Genome British Columbia awards from Genome BC's own funded-project
portfolio ("Search Projects", https://www.genomebc.ca/funding/project-list):
the WordPress `projects` post type (595 posts via `/wp-json/wp/v2/projects`)
plus each project page's info block (project code, Project Leaders,
Institutions, Budget, Program/Competition, Genome Centre(s), Fiscal Year,
Status). Scraped by `scripts/local/genomebc_to_s3.py`; robots.txt allows all
agents. **595 projects, 2007-2026.** No filter: every published project is kept.

**Co-funded projects stay under Genome BC.** The page's "Genome Centre(s)"
field names the programme owner: `Genome British Columbia` (351, Genome BC's
own programmes: SOF, SIP, UPP, GeneSolve, PoC, ...), `Genome Canada` (163,
Genome Canada competitions Genome BC manages/co-funds: Large-Scale Applied
Research, GAPP, S&T Platforms, B&CB, CPHI, ...) and `Partner` (81, Genome BC
partner programmes, incl. company-hosted platforms). `funder_scheme` records
the programme and, when not Genome BC's own, the partner, e.g.
`Genomic Applications Partnership Program (GAPP) (Genome Canada programme)`.
There is no Genome Canada direct ingest in `openalex_awards_raw` (only
priority-0 acknowledgement stubs), so nothing is double-counted.

**Amount:** the page's "Budget" = total project budget in CAD (all
co-funders, not only Genome BC's share), 99.7% coverage.
**Dates:** only the fiscal year funded is published (77%), so
`start_year` = fiscal year and `start_date` stays NULL.

**`funder_award_id` (§2.1.1):** the project code printed on each page
(`SIP012`, `COV016`, `212SEQ`, `DIA013`, ...), which is what citing works
acknowledge: 288 existing Genome BC stub rows (104 ids) collapse onto these
awards; a further 156 Genome Canada stub rows (F4320310787, 53 ids) carry the
same codes for the co-funded Genome Canada projects (left as they are).

**Prerequisites:** run `scripts/local/genomebc_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/genomebc/genomebc_projects.parquet` (§1.4 shrink guard).

**Funder details (F4320* path):**
- funder_id: `4320319973`
- display_name: Genome British Columbia
- ror_id / doi: from `openalex.funders.funders` (ror 03gne5057, doi 10.13039/501100000233)

**Priority:** `594` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.genomebc_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/genomebc/genomebc_projects.parquet`;

In [ ]:
%sql
SELECT genome_centres, status, COUNT(*) as n
FROM openalex.awards.genomebc_raw
GROUP BY 1, 2 ORDER BY 1, 2;

In [ ]:
%sql
DESCRIBE openalex.awards.genomebc_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.genomebc_raw LIMIT 5;

## Step 1.6: Funder existence check (F4320* path)
F4320319973 must resolve to exactly 1 row in `openalex.funders.funders`.
If 0 rows, STOP and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320319973;

## Step 2: Create Genome BC Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.genomebc_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320319973  -- Genome British Columbia
),

people AS (
    -- project leaders in page order (lead first). Genome BC publishes one
    -- institution string per project; it is attached to the lead only.
    -- struct field order MUST match openalex_awards_raw: ...orcid, role_start, affiliation.
    SELECT
        r.wp_id,
        TRANSFORM(
            from_json(r.leaders_json, 'ARRAY<STRUCT<given_name:STRING, family_name:STRING, name:STRING>>'),
            (p, i) -> named_struct(
                'given_name', NULLIF(TRIM(p.given_name), ''),
                'family_name', NULLIF(TRIM(p.family_name), ''),
                'orcid', CAST(NULL AS STRING),
                'role_start', CAST(NULL AS DATE),
                'affiliation', CASE WHEN i = 0 AND NULLIF(TRIM(r.institutions), '') IS NOT NULL THEN named_struct(
                    'name', TRIM(r.institutions),
                    'country', CAST(NULL AS STRING),
                    'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
                ) END
            )
        ) AS investigators
    FROM openalex.awards.genomebc_raw r
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.project_code))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.project_code) as funder_award_id,
    NULLIF(TRY_CAST(g.amount AS DOUBLE), 0) as amount,
    CASE WHEN NULLIF(TRY_CAST(g.amount AS DOUBLE), 0) IS NOT NULL THEN 'CAD' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    'research' as funding_type,
    CASE
        WHEN g.genome_centres = 'Genome Canada' THEN CONCAT(g.program_competition, ' (Genome Canada programme)')
        WHEN g.genome_centres = 'Partner' THEN CONCAT(g.program_competition, ' (partner programme)')
        ELSE g.program_competition
    END as funder_scheme,
    'genomebc_project_portfolio' as provenance,
    CAST(NULL AS DATE) as start_date,
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.fiscal_year AS INT) as start_year,
    CAST(NULL AS INT) as end_year,
    try_element_at(p.investigators, 1) as lead_investigator,
    try_element_at(p.investigators, 2) as co_lead_investigator,
    CASE WHEN size(p.investigators) > 0 THEN p.investigators END as investigators,
    g.landing_page_url as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.project_code))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.genomebc_raw g
JOIN people p ON p.wp_id = g.wp_id
CROSS JOIN src_funder f
WHERE g.project_code IS NOT NULL AND TRIM(g.project_code) != '';

In [ ]:
%sql
-- Shape check (§2.1.1): project codes are short alphanumerics (SIP012, 212SEQ,
-- COV-016, WD21061); expect 0 rows that look like WP ids, URLs or blanks.
SELECT funder_award_id, display_name
FROM openalex.awards.genomebc_awards
WHERE NOT funder_award_id RLIKE '^[A-Za-z0-9][A-Za-z0-9-]{2,11}$';

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'genomebc_project_portfolio' AND priority = 594;

-- Insert into openalex_awards_raw with priority.
-- Priority 594: direct-from-funder ingest of Genome BC's project portfolio.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500), so 594 outranks
-- the acknowledgement shells (priority 0) that share these award ids.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    594 as priority
FROM openalex.awards.genomebc_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_genomebc_awards, COUNT(DISTINCT id) as distinct_ids
FROM openalex.awards.genomebc_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, amount, currency, start_year,
       lead_investigator.given_name, lead_investigator.family_name,
       lead_investigator.affiliation.name, landing_page_url
FROM openalex.awards.genomebc_awards LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_cad
FROM openalex.awards.genomebc_awards
GROUP BY 1 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT start_year, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_cad
FROM openalex.awards.genomebc_awards
GROUP BY start_year ORDER BY start_year;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail.
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.genomebc_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.genomebc_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage (expect ~100% title/description/PI/amount, 77% start_year).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(DISTINCT currency) as distinct_currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_year) as has_start_year,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution
FROM openalex.awards.genomebc_awards;

In [ ]:
%sql
-- Overlap with existing acknowledgement stubs: Genome BC's own, and Genome
-- Canada's (same codes for co-funded Genome Canada projects; not touched).
SELECT c.funder_id, c.provenance, COUNT(*) as stub_rows, COUNT(b.id) as matching_genomebc_award
FROM openalex.awards.openalex_awards_raw c
LEFT JOIN openalex.awards.genomebc_awards b
  ON LOWER(TRIM(b.funder_award_id)) = LOWER(TRIM(c.funder_award_id))
WHERE c.funder_id IN (4320319973, 4320310787) AND c.priority = 0
GROUP BY 1, 2 ORDER BY 1, 2;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'genomebc_project_portfolio'
GROUP BY provenance, priority;